In [ ]:
!pip install -q fastapi uvicorn onnxruntime skl2onnx mlflow scikit-learn pandas numpy requests


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import sys
import subprocess
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ============================================================
# 3. CREATE PROJECT DIRECTORY
# ============================================================

os.makedirs("ml_project", exist_ok=True)


# ============================================================
# 4. CREATE data_loader.py
# ============================================================

data_loader_code = r'''
import pandas as pd
from sklearn.datasets import load_breast_cancer


def load_training_data():

    data = load_breast_cancer()

    X = pd.DataFrame(
        data.data,
        columns=data.feature_names
    )

    y = pd.Series(
        data.target,
        name="target"
    )

    return X, y


def load_prediction_data(data_path):

    return pd.read_csv(data_path)
'''

with open(
    "ml_project/data_loader.py",
    "w"
) as f:
    f.write(data_loader_code)


# ============================================================
# 5. CREATE feature_pipeline.py
# ============================================================

feature_pipeline_code = r'''
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def create_feature_pipeline():

    pipeline = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        )
    ])

    return pipeline
'''

with open(
    "ml_project/feature_pipeline.py",
    "w"
) as f:
    f.write(feature_pipeline_code)


# ============================================================
# 6. CREATE trainer.py
# ============================================================

trainer_code = r'''
import os
import numpy as np
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

from data_loader import load_training_data
from feature_pipeline import create_feature_pipeline


MODEL_DIR = "models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


def train_model():

    X, y = load_training_data()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    pipeline = create_feature_pipeline()

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print(
        "Validation Accuracy:",
        accuracy
    )

    # --------------------------------------------------------
    # MLflow RUN 1
    # --------------------------------------------------------

    with mlflow.start_run(
        run_name="training_run_1"
    ):

        mlflow.log_param(
            "model",
            "LogisticRegression"
        )

        mlflow.log_param(
            "run_number",
            1
        )

        mlflow.log_metric(
            "accuracy",
            accuracy
        )


    # --------------------------------------------------------
    # MLflow RUN 2
    # --------------------------------------------------------

    with mlflow.start_run(
        run_name="training_run_2"
    ):

        mlflow.log_param(
            "model",
            "LogisticRegression"
        )

        mlflow.log_param(
            "run_number",
            2
        )

        mlflow.log_metric(
            "accuracy",
            accuracy
        )


    # --------------------------------------------------------
    # MLflow RUN 3
    # --------------------------------------------------------

    with mlflow.start_run(
        run_name="training_run_3"
    ):

        mlflow.log_param(
            "model",
            "LogisticRegression"
        )

        mlflow.log_param(
            "run_number",
            3
        )

        mlflow.log_metric(
            "accuracy",
            accuracy
        )


    # --------------------------------------------------------
    # Convert complete pipeline to ONNX
    # --------------------------------------------------------

    initial_type = [
        (
            "float_input",
            FloatTensorType(
                [None, X_train.shape[1]]
            )
        )
    ]

    onnx_model = convert_sklearn(
        pipeline,
        initial_types=initial_type,
        options={
            id(pipeline): {
                "zipmap": False
            }
        }
    )

    model_path = os.path.join(
        MODEL_DIR,
        "model.onnx"
    )

    with open(
        model_path,
        "wb"
    ) as f:

        f.write(
            onnx_model.SerializeToString()
        )


    print(
        "ONNX model saved:",
        model_path
    )

    return model_path


if __name__ == "__main__":

    train_model()
'''

with open(
    "ml_project/trainer.py",
    "w"
) as f:
    f.write(trainer_code)


# ============================================================
# 7. CREATE batch_predict.py
# ============================================================

batch_predict_code = r'''
import sys
import numpy as np
import pandas as pd
import onnxruntime as ort


def batch_predict(
    model_path,
    data_path
):

    data = pd.read_csv(
        data_path
    )

    session = ort.InferenceSession(
        model_path,
        providers=["CPUExecutionProvider"]
    )

    input_name = session.get_inputs()[0].name

    X = data.astype(
        np.float32
    ).values

    outputs = session.run(
        None,
        {
            input_name: X
        }
    )

    predictions = outputs[0]
    probabilities = outputs[1]

    # Probability of class 1
    if probabilities.ndim == 2:

        probabilities = probabilities[:, 1]

    predictions = np.asarray(
        predictions
    ).reshape(-1)

    probabilities = np.asarray(
        probabilities
    ).reshape(-1)

    result = data.copy()

    result["prediction"] = predictions.astype(int)

    result["probability"] = probabilities.astype(float)

    result.to_csv(
        "batch_predictions.csv",
        index=False
    )

    return result


if __name__ == "__main__":

    model_path = sys.argv[1]

    data_path = sys.argv[2]

    result = batch_predict(
        model_path,
        data_path
    )

    print(result.head())
'''

with open(
    "ml_project/batch_predict.py",
    "w"
) as f:
    f.write(batch_predict_code)


# ============================================================
# 8. CREATE FASTAPI app.py
# ============================================================

app_code = r'''
from fastapi import FastAPI
from pydantic import BaseModel

import numpy as np
import onnxruntime as ort


app = FastAPI(
    title="ML Prediction API"
)


MODEL_PATH = "models/model.onnx"


session = ort.InferenceSession(
    MODEL_PATH,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name


class PredictionRequest(BaseModel):

    features: list[float]


@app.get("/")
def home():

    return {
        "message": "ML Prediction API is running"
    }


@app.post("/predict")
def predict(
    request: PredictionRequest
):

    X = np.array(
        [request.features],
        dtype=np.float32
    )

    outputs = session.run(
        None,
        {
            input_name: X
        }
    )

    prediction = int(
        np.asarray(
            outputs[0]
        ).reshape(-1)[0]
    )

    probabilities = np.asarray(
        outputs[1]
    )

    if probabilities.ndim == 2:

        probability = float(
            probabilities[0][1]
        )

    else:

        probability = float(
            probabilities.reshape(-1)[0]
        )

    return {
        "prediction": prediction,
        "probability": probability
    }
'''

with open(
    "ml_project/app.py",
    "w"
) as f:
    f.write(app_code)


# ============================================================
# 9. CREATE end_to_end_check.py
# ============================================================

end_to_end_code = r'''
import numpy as np
import pandas as pd
import onnxruntime as ort


def end_to_end_check(
    model_path,
    data_path,
    n_batch=20
):

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    session = ort.InferenceSession(
        model_path,
        providers=["CPUExecutionProvider"]
    )

    input_name = session.get_inputs()[0].name


    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    data = pd.read_csv(
        data_path
    )

    X = data.head(
        n_batch
    ).astype(
        np.float32
    ).values


    # --------------------------------------------------------
    # Make predictions
    # --------------------------------------------------------

    outputs = session.run(
        None,
        {
            input_name: X
        }
    )

    predictions = np.asarray(
        outputs[0]
    ).reshape(-1)

    probabilities = np.asarray(
        outputs[1]
    )


    # --------------------------------------------------------
    # Get probability for class 1
    # --------------------------------------------------------

    if probabilities.ndim == 2:

        probabilities = probabilities[:, 1]

    else:

        probabilities = probabilities.reshape(-1)


    # --------------------------------------------------------
    # Assertions
    # --------------------------------------------------------

    assert len(predictions) == n_batch

    assert len(probabilities) == n_batch

    # Predictions must be 0 or 1
    assert np.all(
        np.isin(
            predictions,
            [0, 1]
        )
    )

    # Probabilities must be between 0 and 1
    assert np.all(
        probabilities >= 0
    )

    assert np.all(
        probabilities <= 1
    )


    print(
        "Pipeline check PASSED"
    )

    print(
        "Number of samples:",
        n_batch
    )

    print(
        "Predictions:",
        predictions.astype(int)
    )

    print(
        "Probabilities:",
        probabilities
    )


    return True
'''

with open(
    "ml_project/end_to_end_check.py",
    "w"
) as f:
    f.write(end_to_end_code)


# ============================================================
# 10. ADD PROJECT DIRECTORY TO PYTHON PATH
# ============================================================

sys.path.insert(
    0,
    os.path.abspath("ml_project")
)


# ============================================================
# 11. TRAIN MODEL
# ============================================================

from trainer import train_model

model_path = train_model()


# ============================================================
# 12. CREATE SAMPLE CSV FOR BATCH PREDICTION
# ============================================================

from data_loader import load_training_data

X_data, y_data = load_training_data()

prediction_data = X_data.head(
    50
)

prediction_data.to_csv(
    "prediction_data.csv",
    index=False
)

print(
    "\nPrediction data saved as prediction_data.csv"
)


# ============================================================
# 13. RUN BATCH PREDICTION
# ============================================================

from batch_predict import batch_predict

batch_result = batch_predict(
    model_path,
    "prediction_data.csv"
)

print("\nBatch Prediction Results:")
print(
    batch_result.head(10)
)


# ============================================================
# 14. RUN END-TO-END CHECK
# ============================================================

from end_to_end_check import end_to_end_check

end_to_end_check(
    model_path,
    "prediction_data.csv",
    n_batch=20
)


# ============================================================
# 15. FASTAPI PREDICT TEST
# ============================================================

from fastapi.testclient import TestClient

from app import app

client = TestClient(
    app
)

# Take one sample
sample_features = X_data.iloc[
    0
].astype(
    float
).tolist()


response = client.post(
    "/predict",
    json={
        "features": sample_features
    }
)

print("\nFastAPI /predict Response:")
print(
    response.json()
)


# ============================================================
# 16. FASTAPI RESPONSE ASSERTIONS
# ============================================================

api_result = response.json()

assert "prediction" in api_result

assert "probability" in api_result

assert api_result["prediction"] in [
    0,
    1
]

assert (
    0 <=
    api_result["probability"] <=
    1
)

print(
    "\nFastAPI prediction check PASSED"
)


# ============================================================
# 17. FINAL FILES
# ============================================================

print("\n================================================")
print("PROJECT FILES")
print("================================================")

print("data_loader.py")
print("feature_pipeline.py")
print("trainer.py")
print("batch_predict.py")
print("app.py")
print("end_to_end_check.py")
print("models/model.onnx")
print("prediction_data.csv")
print("batch_predictions.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 756.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026/10/05 08:36:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/05 08:36:35 INFO mlflow.store.db.utils: Updating database tables


ONNX model saved: models/model.onnx

Prediction data saved as prediction_data.csv

Batch Prediction Results:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   
5        12.45         15.70           82.57      477.1          0.12780   
6        18.25         19.98          119.60     1040.0          0.09463   
7        13.71         20.83           90.20      577.9          0.11890   
8        13.00         21.82           87.50      519.8          0.12730   
9        12.46         24.04           83.97      475.9          0.11860   

   mean compactness  mean concavity  mean concave poin